# Setup: EvalHub Service Deployment & SDK Configuration

This notebook deploys the [EvalHub](https://github.com/eval-hub/eval-hub) service on OpenShift and configures the [eval-hub-sdk](https://github.com/eval-hub/eval-hub-sdk) to run LLM evaluations (including [lm-evaluation-harness](https://github.com/EleutherAI/lm-evaluation-harness)) through a centralized REST API with **MLflow experiment tracking**.

## What is EvalHub?

EvalHub is a lightweight REST API service that orchestrates LLM evaluations across multiple backends. It:

- Routes evaluation requests to frameworks like **lm-evaluation-harness**, RAGAS, GuideLLM, LightEval, and more
- Tracks experiments via **MLflow** (metrics, parameters, artifacts)
- Runs natively on **OpenShift** via the TrustyAI Operator
- Supports a **"Bring Your Own Framework" (BYOF)** approach through the SDK

## EvalHub vs. LMEvalJob (1_LMEval_setup.ipynb)

| Feature | LMEvalJob (Phase 1) | EvalHub (Phase 2) |
|---------|--------------------|---------|
| Interface | Kubernetes CR (YAML) | REST API + Python SDK |
| Frameworks | lm-evaluation-harness only | Multiple (lm-eval, RAGAS, LightEval, ...) |
| Experiment tracking | Manual | Built-in MLflow integration |
| Multi-benchmark jobs | One task per CR | Multiple benchmarks per request |
| Result management | Pod logs / CR status | Centralized API + MLflow UI |

## Prerequisites

- Completed **0_model_deploy.ipynb** (model deployed on OpenShift AI)
- Completed **1_LMEval_setup.ipynb** (RBAC and secrets configured)
- TrustyAI Operator installed on the cluster

---

## Part A: Deploy EvalHub Service on OpenShift

Before using the SDK, the EvalHub service and MLflow must be running on the cluster. This section walks through the deployment.

### Step A-1: Configuration

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv(dotenv_path="../.env")

NAMESPACE = os.getenv("NAMESPACE", "hyo-project")

print(f"Namespace: {NAMESPACE}")

### Step A-2: Verify TrustyAI Operator

The TrustyAI Operator manages the `EvalHub` Custom Resource. Verify it is installed on the cluster:

In [ ]:
!oc get csv -n openshift-operators | grep trustyai || \
    echo "TrustyAI Operator not found. Install it from OperatorHub first."

### Step A-3: Deploy MLflow (if not already running)

EvalHub requires an MLflow tracking server to store experiment metrics and artifacts. If MLflow is not yet deployed, create one:

> **Note:** If MLflow is already running on your cluster, skip this step and note the service URL (e.g., `http://mlflow.my-namespace.svc.cluster.local:5000`).

In [ ]:
mlflow_yaml = f"""apiVersion: apps/v1
kind: Deployment
metadata:
  name: mlflow
  namespace: {NAMESPACE}
spec:
  replicas: 1
  selector:
    matchLabels:
      app: mlflow
  template:
    metadata:
      labels:
        app: mlflow
    spec:
      containers:
        - name: mlflow
          image: ghcr.io/mlflow/mlflow:v2.22.0
          command: ["mlflow", "server"]
          args:
            - "--host=0.0.0.0"
            - "--port=5000"
            - "--backend-store-uri=sqlite:///mlflow/mlflow.db"
            - "--default-artifact-root=/mlflow/artifacts"
          ports:
            - containerPort: 5000
          volumeMounts:
            - name: mlflow-data
              mountPath: /mlflow
      volumes:
        - name: mlflow-data
          persistentVolumeClaim:
            claimName: mlflow-pvc
---
apiVersion: v1
kind: PersistentVolumeClaim
metadata:
  name: mlflow-pvc
  namespace: {NAMESPACE}
spec:
  accessModes: [ReadWriteOnce]
  resources:
    requests:
      storage: 5Gi
---
apiVersion: v1
kind: Service
metadata:
  name: mlflow
  namespace: {NAMESPACE}
spec:
  selector:
    app: mlflow
  ports:
    - port: 5000
      targetPort: 5000
"""

with open("/tmp/mlflow-deploy.yaml", "w") as f:
    f.write(mlflow_yaml)

print("MLflow deployment YAML generated.")
print("Review and apply with the next cell.")

In [ ]:
# Uncomment to deploy MLflow:
# !oc apply -f /tmp/mlflow-deploy.yaml
# !oc rollout status deployment/mlflow -n {NAMESPACE} --timeout=120s

### Step A-4: Deploy EvalHub via the TrustyAI Operator

Create an `EvalHub` Custom Resource. The TrustyAI Operator will reconcile it into a running EvalHub service with the configured MLflow connection.

Key fields:
- `MLFLOW_TRACKING_URI` — Points to your MLflow service
- `replicas` — Number of EvalHub instances

In [ ]:
MLFLOW_SVC_URL = f"http://mlflow.{NAMESPACE}.svc.cluster.local:5000"

evalhub_cr_yaml = f"""apiVersion: trustyai.opendatahub.io/v1alpha1
kind: EvalHub
metadata:
  name: evalhub
  namespace: {NAMESPACE}
spec:
  replicas: 1
  env:
    - name: MLFLOW_TRACKING_URI
      value: "{MLFLOW_SVC_URL}"
"""

with open("/tmp/evalhub-cr.yaml", "w") as f:
    f.write(evalhub_cr_yaml)

print("EvalHub CR YAML:")
print(evalhub_cr_yaml)

In [ ]:
!oc apply -f /tmp/evalhub-cr.yaml

### Step A-5: Verify EvalHub Deployment

Wait for the EvalHub pod to become ready and check its status:

In [ ]:
!oc get evalhub -n {NAMESPACE}
print()
!oc get pods -n {NAMESPACE} | grep -E "evalhub|mlflow"

### Step A-6: Get the EvalHub Service URL

The EvalHub service URL is constructed from the Kubernetes Service created by the operator. There are two ways to obtain it:

1. **Cluster-internal** (from a Workbench pod): `http://evalhub.<namespace>.svc.cluster.local:8080`
2. **Via Route** (if an OpenShift Route is created): external URL exposed by the cluster

In [ ]:
import subprocess, json

# Method 1: Get cluster-internal service URL
result = subprocess.run(
    ["oc", "get", "svc", "-n", NAMESPACE, "-l", "app=evalhub",
     "-o", "jsonpath={.items[0].metadata.name}"],
    capture_output=True, text=True,
)
svc_name = result.stdout.strip() or "evalhub"
EVALHUB_INTERNAL_URL = f"http://{svc_name}.{NAMESPACE}.svc.cluster.local:8080"

# Method 2: Check for an OpenShift Route
result_route = subprocess.run(
    ["oc", "get", "route", "-n", NAMESPACE, "-l", "app=evalhub",
     "-o", "jsonpath={.items[0].spec.host}"],
    capture_output=True, text=True,
)
route_host = result_route.stdout.strip()

print("EvalHub Service Discovery")
print("=" * 60)
print(f"  Service name:         {svc_name}")
print(f"  Cluster-internal URL: {EVALHUB_INTERNAL_URL}")
if route_host:
    EVALHUB_ROUTE_URL = f"https://{route_host}"
    print(f"  External Route URL:   {EVALHUB_ROUTE_URL}")
else:
    print(f"  External Route:       (none — create one if external access is needed)")

print(f"\n  Use the cluster-internal URL in .env as EVALHUB_URL")
print(f"  e.g., EVALHUB_URL={EVALHUB_INTERNAL_URL}")

### Step A-7: Health Check

Verify the EvalHub service is responding before proceeding to SDK setup:

In [ ]:
!curl -sk {EVALHUB_INTERNAL_URL}/api/v1/health | python3 -m json.tool

---

## Part B: Configure the EvalHub SDK

Now that the EvalHub service is running, install and configure the Python SDK.

### Step B-1: Install the EvalHub SDK

The `eval-hub-sdk` package provides both a Python client for submitting evaluations and the adapter SDK for building custom frameworks.

In [ ]:
!pip install -q eval-hub-sdk

### Step B-2: Load Configuration

Configuration is loaded from `../.env`. Update these EvalHub-specific variables with the values discovered in Part A:

| Variable | Description | Example |
|----------|-------------|---------|
| `EVALHUB_URL` | EvalHub service endpoint (from Step A-6) | `http://evalhub.my-namespace.svc.cluster.local:8080` |
| `EVALHUB_AUTH_TOKEN` | Authentication token (optional) | SA token or API key |
| `MLFLOW_TRACKING_URI` | MLflow server URL (from Step A-3) | `http://mlflow.my-namespace.svc.cluster.local:5000` |

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv(dotenv_path="../.env")

NAMESPACE = os.getenv("NAMESPACE", "hyo-project")
MODEL_NAME = os.getenv("MODEL_NAME", "vllm-gemma4-e2b")
BASE_URL = os.getenv("BASE_URL", f"https://{MODEL_NAME}-predictor.{NAMESPACE}.svc.cluster.local:8443/v1")
EVALHUB_URL = os.getenv("EVALHUB_URL", "http://evalhub:8080")
EVALHUB_AUTH_TOKEN = os.getenv("EVALHUB_AUTH_TOKEN", None)
MLFLOW_TRACKING_URI = os.getenv("MLFLOW_TRACKING_URI", "http://mlflow:5000")

print(f"Namespace:          {NAMESPACE}")
print(f"Model Name:         {MODEL_NAME}")
print(f"Model Endpoint:     {BASE_URL}")
print(f"EvalHub URL:        {EVALHUB_URL}")
print(f"Auth Token:         {'***' if EVALHUB_AUTH_TOKEN else 'None (no auth)'}")
print(f"MLflow Tracking:    {MLFLOW_TRACKING_URI}")

### Step B-3: Verify EvalHub Connectivity

Check that the SDK can connect to the EvalHub service.

In [ ]:
from evalhub import SyncEvalHubClient

client = SyncEvalHubClient(
    base_url=EVALHUB_URL,
    auth_token=EVALHUB_AUTH_TOKEN,
    insecure=True,
    tenant=NAMESPACE,
)

print(f"Connected to EvalHub at {EVALHUB_URL}")
print(f"Tenant (namespace):  {NAMESPACE}")

### Step B-4: Explore Available Providers and Benchmarks

EvalHub ships with pre-configured providers. Let's list them and their benchmarks.

In [ ]:
providers = client.providers.list()

print(f"Available Providers ({providers.total_count}):")
print("=" * 60)
for provider in providers.items:
    print(f"\n  Provider: {provider.name}")
    print(f"  ID:       {provider.resource.id}")
    print(f"  Desc:     {provider.description}")
    print(f"  Benchmarks: {len(provider.benchmarks)}")

In [ ]:
benchmarks = client.benchmarks.list()

print(f"\nAvailable Benchmarks ({benchmarks.total_count}):")
print("=" * 60)
for bm in benchmarks.items[:20]:
    print(f"  {bm.id:30s}  category={bm.category or 'N/A':15s}  metrics={bm.metrics}")

if benchmarks.total_count > 20:
    print(f"  ... and {benchmarks.total_count - 20} more")

### Filter Korean Benchmarks

Look for Korean-specific benchmarks available through lm-evaluation-harness.

In [ ]:
korean_keywords = ["kmmlu", "kobest", "haerae", "klue", "korean", "ko_"]

korean_benchmarks = [
    bm for bm in benchmarks.items
    if any(kw in bm.id.lower() for kw in korean_keywords)
]

print(f"Korean Benchmarks Found: {len(korean_benchmarks)}")
print("=" * 60)
for bm in korean_benchmarks:
    print(f"  {bm.id:35s}  {bm.name}")

### Step B-5: Configure the Model Endpoint

The `ModelConfig` specifies which model endpoint EvalHub should target. This points to your deployed vLLM InferenceService.

#### Key Parameters

| Parameter | Description | Example |
|-----------|-------------|---------|
| `url` | OpenAI-compatible endpoint URL | `https://model-predictor.ns.svc:8443/v1` |
| `name` | Model name (as registered in vLLM) | `vllm-gemma4-e2b` |
| `auth.secret_ref` | K8s Secret for model auth (optional) | `lmeval-sa-token` |

In [ ]:
from evalhub import ModelConfig

model = ModelConfig(
    url=BASE_URL,
    name=MODEL_NAME,
)

print("Model Configuration:")
print(f"  URL:   {model.url}")
print(f"  Name:  {model.name}")
print(f"  Auth:  {model.auth or 'None (using cluster-internal access)'}")

#### (Optional) Model Authentication

If your InferenceService has OAuth enabled (`security.opendatahub.io/enable-auth: "true"`), reference a Kubernetes Secret containing the ServiceAccount token:

In [ ]:
from evalhub.models.api import ModelAuth

model_with_auth = ModelConfig(
    url=BASE_URL,
    name=MODEL_NAME,
    auth=ModelAuth(secret_ref="lmeval-sa-token"),
)

print("Model Configuration (with auth):")
print(f"  URL:        {model_with_auth.url}")
print(f"  Name:       {model_with_auth.name}")
print(f"  Auth:       secret_ref={model_with_auth.auth.secret_ref}")

### Step B-6: Configure MLflow Experiment Tracking

EvalHub integrates with MLflow to automatically track evaluation metrics, parameters, and artifacts. When you include an `ExperimentConfig` in your job submission, EvalHub will:

1. Create (or reuse) an MLflow experiment with the given name
2. Log all benchmark metrics (accuracy, f1, etc.) as MLflow metrics
3. Tag the run with model info, benchmark details, and custom tags
4. Store detailed result artifacts

The MLflow connection was configured in **Step A-4** via `MLFLOW_TRACKING_URI` in the EvalHub CR.

#### ExperimentConfig in Job Submission

You control experiment tracking per-job via the `experiment` field:

In [ ]:
from evalhub import ExperimentConfig, ExperimentTag

experiment = ExperimentConfig(
    name="korean-llm-evaluation",
    tags=[
        ExperimentTag(key="model_family", value="gemma-4"),
        ExperimentTag(key="language", value="korean"),
        ExperimentTag(key="environment", value="dev"),
        ExperimentTag(key="team", value="ai-evaluation"),
    ],
)

print("MLflow Experiment Configuration:")
print(f"  Name:  {experiment.name}")
print(f"  Tags:")
for tag in experiment.tags:
    print(f"    {tag.key}: {tag.value}")

### Step B-7: Submit a Single Benchmark Evaluation

Let's submit a simple evaluation using the `lm_evaluation_harness` provider with a Korean benchmark.

In [ ]:
from evalhub import BenchmarkConfig, JobSubmissionRequest

single_job_request = JobSubmissionRequest(
    name="kmmlu-law-evaluation",
    description="Korean MMLU Law benchmark via lm-evaluation-harness",
    tags=["korean", "kmmlu", "law"],
    model=model,
    benchmarks=[
        BenchmarkConfig(
            id="kmmlu_direct_law",
            provider_id="lm_evaluation_harness",
            parameters={
                "num_fewshot": 0,
                "limit": 5,
            },
        ),
    ],
    experiment=experiment,
)

print("Job Submission Request:")
print(f"  Name:       {single_job_request.name}")
print(f"  Model:      {single_job_request.model.name} @ {single_job_request.model.url}")
print(f"  Benchmarks: {[b.id for b in single_job_request.benchmarks]}")
print(f"  Experiment: {single_job_request.experiment.name}")

In [ ]:
job = client.jobs.submit(single_job_request)

print(f"Job submitted!")
print(f"  Job ID:        {job.id}")
print(f"  State:         {job.state}")
print(f"  MLflow Exp ID: {job.resource.mlflow_experiment_id or 'pending'}")

### Step B-8: Monitor Job Progress

Poll the job status until it completes.

In [ ]:
import time
from evalhub import JobStatus

TERMINAL_STATES = {JobStatus.COMPLETED, JobStatus.FAILED, JobStatus.CANCELLED, JobStatus.PARTIALLY_FAILED}

print(f"Monitoring job {job.id}...")
print("-" * 60)

while True:
    status = client.jobs.get(job.id)
    state = status.effective_state
    
    msg = ""
    if status.status and status.status.message:
        msg = f" - {status.status.message.message}"
    print(f"  [{state.value:>10s}]{msg}")

    if state in TERMINAL_STATES:
        break

    time.sleep(10)

print("-" * 60)
print(f"Final state: {state.value}")

### Step B-9: View Results

Retrieve the evaluation results, including MLflow run information.

In [ ]:
completed_job = client.jobs.get(job.id)

if completed_job.results:
    print("Evaluation Results:")
    print("=" * 60)

    if completed_job.results.mlflow_experiment_url:
        print(f"\n  MLflow Experiment: {completed_job.results.mlflow_experiment_url}")

    for bm_result in completed_job.results.benchmarks:
        print(f"\n  Benchmark: {bm_result.id} (provider: {bm_result.provider_id})")
        if bm_result.mlflow_run_id:
            print(f"  MLflow Run ID: {bm_result.mlflow_run_id}")
        print(f"  Metrics:")
        for metric_name, metric_value in bm_result.metrics.items():
            print(f"    {metric_name}: {metric_value}")
else:
    print("No results available yet.")

### Step B-10: Multi-Benchmark Evaluation

Submit multiple benchmarks in a single request. EvalHub runs them concurrently and tracks all results under one MLflow experiment.

In [ ]:
multi_job_request = JobSubmissionRequest(
    name="korean-comprehensive-eval",
    description="Multi-benchmark Korean LLM evaluation",
    tags=["korean", "comprehensive"],
    model=model,
    benchmarks=[
        BenchmarkConfig(
            id="kmmlu_direct_law",
            provider_id="lm_evaluation_harness",
            parameters={"num_fewshot": 0, "limit": 5},
        ),
        BenchmarkConfig(
            id="kobest_wic",
            provider_id="lm_evaluation_harness",
            parameters={"num_fewshot": 0, "limit": 5},
        ),
        BenchmarkConfig(
            id="arc_easy",
            provider_id="lm_evaluation_harness",
            parameters={"num_fewshot": 0, "limit": 5},
        ),
    ],
    experiment=ExperimentConfig(
        name="korean-comprehensive-evaluation",
        tags=[
            ExperimentTag(key="evaluation_type", value="comprehensive"),
            ExperimentTag(key="model_family", value="gemma-4"),
            ExperimentTag(key="language", value="korean,english"),
        ],
    ),
)

print("Multi-Benchmark Job Request:")
print(f"  Name:       {multi_job_request.name}")
print(f"  Benchmarks: {[b.id for b in multi_job_request.benchmarks]}")
print(f"  Experiment: {multi_job_request.experiment.name}")

# Uncomment to submit:
# multi_job = client.jobs.submit(multi_job_request)
# print(f"\nJob submitted: {multi_job.id}")

### Step B-11: Use Collections for Standardized Evaluations

Collections group benchmarks into reusable evaluation suites. This is useful for certification or compliance workflows.

In [ ]:
collections = client.collections.list()

print(f"Available Collections ({collections.total_count}):")
print("=" * 60)
for coll in collections.items:
    print(f"\n  Collection: {coll.name}")
    print(f"  ID:         {coll.resource.id}")
    print(f"  Category:   {coll.category}")
    print(f"  Benchmarks: {len(coll.benchmarks)}")
    for bm_ref in coll.benchmarks[:5]:
        print(f"    - {bm_ref.id} (provider: {bm_ref.provider_id})")
    if len(coll.benchmarks) > 5:
        print(f"    ... and {len(coll.benchmarks) - 5} more")

### Step B-12: List and Manage Jobs

Review all submitted evaluation jobs.

In [ ]:
jobs_list = client.jobs.list()

print(f"Evaluation Jobs ({jobs_list.total_count}):")
print("=" * 60)
for j in jobs_list.items:
    state = j.effective_state.value
    exp_name = j.experiment.name if j.experiment else "N/A"
    benchmarks = [b.id for b in j.benchmarks] if j.benchmarks else []
    print(f"  [{state:>16s}] {j.id[:12]}... | {j.name} | exp={exp_name} | benchmarks={benchmarks}")

## Reference: EvalHub SDK Quick Reference

### Client SDK Imports

```python
from evalhub import (
    SyncEvalHubClient,          # Synchronous client (recommended for notebooks)
    AsyncEvalHubClient,         # Async client (for production apps)
    ModelConfig,                # Model endpoint configuration
    BenchmarkConfig,            # Benchmark selection and parameters
    JobSubmissionRequest,       # Full job request
    ExperimentConfig,           # MLflow experiment settings
    ExperimentTag,              # MLflow tags
    CollectionRef,              # Reference to a benchmark collection
    EvaluationExports,          # OCI artifact export config
    EvaluationExportsOCI,       # OCI-specific export settings
    OCICoordinates,             # OCI registry coordinates
    JobStatus,                  # Job status enum
)
```

### Key API Patterns

```python
# Initialize client
client = SyncEvalHubClient(
    base_url="http://evalhub:8080",
    auth_token="...",           # Optional: SA token or API key
    insecure=True,              # Skip TLS verification (dev only)
    tenant="my-namespace",      # Kubernetes namespace
)

# Explore resources
providers  = client.providers.list()
benchmarks = client.benchmarks.list()
collections = client.collections.list()

# Submit a job
job = client.jobs.submit(request)

# Monitor and retrieve results
status = client.jobs.get(job.id)
```

### MLflow Experiment Structure

When an `ExperimentConfig` is provided:

- **Experiment Name**: `{prefix}_{experiment.name}`
- **Tags**: Direct mapping from `experiment.tags`
- **Run**: One MLflow run per evaluation request
- **Metrics**: Benchmark scores logged automatically
- **Parameters**: Model config and benchmark settings logged
- **Artifacts**: Detailed result files stored

### Useful Links

- [EvalHub GitHub](https://github.com/eval-hub/eval-hub)
- [EvalHub SDK GitHub](https://github.com/eval-hub/eval-hub-sdk)
- [EvalHub API Docs](https://eval-hub.github.io/eval-hub/)
- [MLflow Integration Guide](https://github.com/eval-hub/eval-hub/blob/main/MLFLOW.md)

## Done!

You've now configured the EvalHub SDK and learned how to:

1. **Connect** to the EvalHub service with the Python SDK
2. **Configure a model endpoint** pointing to your deployed InferenceService
3. **Set up MLflow experiment tracking** with tags and experiment names
4. **Submit evaluations** using lm-evaluation-harness benchmarks
5. **Monitor** job progress and **retrieve results**
6. **Run multi-benchmark** evaluations in a single request

### Next Steps

- **1_builtin_tasks/** -- Run quick evaluations using LMEvalJob (Kubernetes CR)
- **2_custom_tasks/** -- Create custom evaluation tasks with Git-sourced datasets
- **4_eval_hub_benchmark/** -- Analyze benchmark results tracked in EvalHub + MLflow